In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print("Working dir:", os.getcwd())

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split

# Adjust path below to where your step7_pca.csv is located in your Drive
RAW_PATH = "/content/drive/MyDrive/Colab Notebooks/data/processed/Dataset7.csv"
if not os.path.exists(RAW_PATH):
    # fallback: try current folder
    RAW_PATH = "Dataset7.csv"

df = pd.read_csv(RAW_PATH)
print("Loaded:", RAW_PATH, "shape:", df.shape)
display(df.head())
print("\nColumns:", df.columns.tolist())


In [ ]:
# Cell 2: Prepare features and target (shared)
from sklearn.model_selection import train_test_split

# Ensure target exists
assert 'selling_price' in df.columns, "selling_price not in dataset"

X = df.drop(columns=['selling_price'])
y = df['selling_price']

# If any NaNs remain, fill with median for modeling convenience
X = X.fillna(X.median())

# Train-test split (same for all members to ensure fair comparison)
RANDOM_SEED = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)


In [ ]:
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

# SVR sensitive to scaling: ensure X is scaled
scaler_for_svr = StandardScaler()
X_train_s = scaler_for_svr.fit_transform(X_train)
X_test_s = scaler_for_svr.transform(X_test)

kernels = ['linear', 'poly', 'rbf']
svr_results = []
for k in kernels:
    svr = SVR(kernel=k)
    svr.fit(X_train_s, y_train)
    yp = svr.predict(X_test_s)
    svr_results.append((k, r2_score(y_test, yp), mean_absolute_error(y_test, yp)))

for k, r2v, mae in svr_results:
    print(f"Kernel={k} -> R2: {r2v:.3f}, MAE: {mae:.0f}")

# Visual: pick best kernel by R2 and plot actual vs predicted
best_kernel = max(svr_results, key=lambda t: t[1])[0]
svr_best = SVR(kernel=best_kernel)
svr_best.fit(X_train_s, y_train)
y_pred_svr = svr_best.predict(X_test_s)

plt.figure(figsize=(6,5))
plt.scatter(y_test, y_pred_svr, alpha=0.5, s=20)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.title(f"SVR (kernel={best_kernel}) Actual vs Predicted")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.show()

print("SVR can capture nonlinear relationships with RBF/poly kernels but needs careful scaling and parameter tuning.")